# E6 | Model Clustering K-Means
Segmentar incidentes em 4 clusters (A/B/C/D) para atuacao preventiva

In [10]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np, tempfile, joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


In [11]:
RDS_HOST     = os.getenv('RDS_HOST')
RDS_PORT     = int(os.getenv('RDS_PORT', '5432'))
RDS_USER     = os.getenv('RDS_USER', 'postgres')
RDS_PASSWORD = os.getenv('RDS_PASSWORD')
RDS_DATABASE = os.getenv('RDS_DATABASE', 'aiops_gold')

print(f'Host: {RDS_HOST}')
print(f'DB:   {RDS_DATABASE}')
print(f'User: {RDS_USER}')

connection_url = f"postgresql+psycopg2://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}"

%load_ext sql
%sql {connection_url}

Host: terraform-20260518150028461700000001.c4xegmk24lg6.us-east-1.rds.amazonaws.com
DB:   aiops_gold
User: postgres
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [12]:

from sqlalchemy import create_engine
# Reutilizar credenciais da célula anterior (RDS_HOST, RDS_USER, RDS_PASSWORD, RDS_DATABASE)
engine = create_engine(f"postgresql://{RDS_USER}:{RDS_PASSWORD}@{RDS_HOST}:{RDS_PORT}/{RDS_DATABASE}")

df = pd.read_sql('''SELECT * FROM gold_ml.ml_cluster_dataset''',engine)

In [13]:
# ===== [3b] EDA: ANALISE DOS DADOS ANTES DE TRANSFORMACAO =====
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# CAUSA: Features de ENTRADA
cause_cols = [
    'prioridade_num', 'grupo_designado', 'categoria', 'subcategoria', 'produto',
    'hora_abertura', 'turno_abertura', 'dia_semana_num', 'fora_horario_comercial',
    'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'triagem_incompleta'
]

# EFEITO: Features de SAIDA (EXCLUIR do treino)
effect_cols = [
    'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado',
    'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional'
]

# EXCLUIR: NULL placeholders, redundantes
exclude_cols = [
    'incident_id', 'cluster', 'data_abertura',
    'duracao_horas_scaled', 'cluster_id',
    'is_filho_de_problema'
]

print('[STATS] ESTRUTURA DE FEATURES:')
print('   [OK] CAUSA (incluir): %d colunas' % len(cause_cols))
print('   [WARN] EFEITO (interpretar): %d colunas' % len(effect_cols))
print('   [ERROR] EXCLUIR: %d colunas' % len(exclude_cols))
print('   Total dataset: %d' % len(df.columns))

# EDA com 6 graficos
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('EDA: Analise de Features ANTES de Transformacao', fontsize=14, fontweight='bold')

# [1] Cardinalidade
cat_cols_all = df.select_dtypes(include=['object']).columns.tolist()
cardinality = {col: df[col].nunique() for col in cat_cols_all}
cardinality_sorted = dict(sorted(cardinality.items(), key=lambda x: x[1], reverse=True))
axes[0, 0].barh(list(cardinality_sorted.keys()), list(cardinality_sorted.values()), color='steelblue')
axes[0, 0].set_xlabel('Numero de Valores Unicos')
axes[0, 0].set_title('1. Cardinalidade - Colunas Categoricas')
axes[0, 0].grid(axis='x', alpha=0.3)

# [2] Distribuicao
cause_numeric = [c for c in cause_cols if c in df.columns and df[c].dtype in ['int64', 'float64']]
sample_cause = cause_numeric[:4] if len(cause_numeric) >= 4 else cause_numeric
if sample_cause:
    df[sample_cause].hist(ax=axes[0, 1], bins=30, color='green', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('2. Distribuicao - Features CAUSA (amostra)')
axes[0, 1].grid(alpha=0.3)

# [3] Correlacao
if len(cause_numeric) > 1:
    corr_matrix = df[cause_numeric].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
                ax=axes[0, 2], cbar_kws={'label': 'Correlacao'})
axes[0, 2].set_title('3. Correlacao - Features CAUSA')

# [4] NULL values
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
if len(null_pct) > 0:
    axes[1, 0].barh(null_pct.index, null_pct.values, color='coral')
    axes[1, 0].set_xlabel('Percentual de NULLs')
axes[1, 0].set_title('4. Valores NULL')
axes[1, 0].grid(axis='x', alpha=0.3)

# [5] Categoria distribution
if 'categoria' in df.columns:
    categoria_dist = df['categoria'].value_counts().head(15)
    axes[1, 1].barh(categoria_dist.index, categoria_dist.values, color='mediumseagreen')
    axes[1, 1].set_xlabel('Contagem')
axes[1, 1].set_title('5. Top-15 Categorias')
axes[1, 1].grid(axis='x', alpha=0.3)

# [6] Summary
axes[1, 2].axis('off')
summary = ('RESUMO DE FEATURES:\n\n'
          '[OK] CAUSA: %d colunas\n'
          '     Objetivo: Perfil entrada\n\n'
          '[WARN] EFEITO: %d colunas\n'
          '      Objetivo: Interpretar\n\n'
          '[ERROR] EXCLUIR: %d\n'
          '        Data leakage/NULL') % (len(cause_cols), len(effect_cols), len(exclude_cols))
axes[1, 2].text(0.05, 0.95, summary, transform=axes[1, 2].transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 2].set_title('6. Estrategia de Feature Selection', fontweight='bold')

plt.tight_layout()
eda_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'eda_feature_selection.png')
Path(eda_path).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(eda_path, dpi=100, bbox_inches='tight')
plt.close()

print('[OK] EDA salva em: %s' % eda_path)


[STATS] ESTRUTURA DE FEATURES:
   [OK] CAUSA (incluir): 14 colunas
   [WARN] EFEITO (interpretar): 7 colunas
   [ERROR] EXCLUIR: 6 colunas
   Total dataset: 26
[OK] EDA salva em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans\eda_feature_selection.png


In [14]:
# ===== [4] FEATURE ENGINEERING: SELECAO CAUSA + ENCODING OTIMIZADO =====
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA

print('[INFO] FEATURE ENGINEERING: CAUSA + ENCODING OTIMIZADO')

# ===== OUTLIER REMOVAL (OR LOGICO) =====
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['incident_id', 'cluster', 'data_abertura']]

print('[STATS] Dataset Original: %d incidentes' % len(df))

# Outlier detection com OR logico
outlier_counts = pd.DataFrame(index=df.index)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_counts[col] = ~((df[col] >= lower) & (df[col] <= upper))

outlier_total_per_row = outlier_counts.sum(axis=1)
max_outliers_allowed = len(numeric_cols) * 0.5
outlier_mask = outlier_total_per_row <= max_outliers_allowed

df_clean = df[outlier_mask].copy()
n_removed = len(df) - len(df_clean)

print('[STATS] Outlier Detection - Removidos: %d (%.2f%%)' % (n_removed, n_removed/len(df)*100))
print('        Mantidos: %d (%.2f%%)' % (len(df_clean), len(df_clean)/len(df)*100))

# ===== SELECIONAR FEATURES CAUSA =====
cause_cols = [
    'prioridade_num', 'grupo_designado', 'categoria', 'subcategoria', 'produto',
    'hora_abertura', 'turno_abertura', 'dia_semana_num', 'fora_horario_comercial',
    'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'triagem_incompleta'
]

cause_cols_valid = [c for c in cause_cols if c in df_clean.columns]
print('[OK] Features CAUSA validas: %d' % len(cause_cols_valid))

df_features = df_clean[cause_cols_valid].copy()

# ===== ENCODING CICLICO =====
print('[INFO] Encoding Ciclico para Features Temporais')
temporal_cyclic = {
    'hora_abertura': 24,
    'dia_semana_num': 7,
    'mes_abertura': 12
}

for col, period in temporal_cyclic.items():
    if col in df_features.columns:
        df_features[f'{col}_sin'] = np.sin(2 * np.pi * df_features[col] / period)
        df_features[f'{col}_cos'] = np.cos(2 * np.pi * df_features[col] / period)
        df_features = df_features.drop(columns=[col])

# ===== FREQUENCY ENCODING =====
print('[INFO] Frequency Encoding para Alta Cardinalidade')
high_cardinality_cols = ['grupo_designado', 'subcategoria', 'produto']

for col in high_cardinality_cols:
    if col in df_features.columns:
        freq_map = df_features[col].value_counts(normalize=True).to_dict()
        df_features[col] = df_features[col].map(freq_map).fillna(0)

# ===== ONE-HOT ENCODING =====
print('[INFO] One-Hot Encoding para Baixa Cardinalidade')
low_cardinality_cols = ['categoria', 'turno_abertura']
low_cardinality_cols = [c for c in low_cardinality_cols if c in df_features.columns]

if low_cardinality_cols:
    for col in low_cardinality_cols:
        df_features[col] = df_features[col].fillna('unknown').astype(str)
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', max_categories=50)
    X_ohe = encoder.fit_transform(df_features[low_cardinality_cols])
    feature_names_ohe = encoder.get_feature_names_out(low_cardinality_cols)
    X_ohe_df = pd.DataFrame(X_ohe, columns=feature_names_ohe, index=df_features.index)
    
    df_features = df_features.drop(columns=low_cardinality_cols)
    df_features = pd.concat([df_features, X_ohe_df], axis=1)

# ===== STANDARDSCALER =====
print('[INFO] StandardScaler Normalization')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features.fillna(0))

reduction_pct = (1 - X_scaled.shape[1]/680)*100
print('[OK] Features Finais: %d (reducao de %.1f%%)' % (X_scaled.shape[1], reduction_pct))


[INFO] FEATURE ENGINEERING: CAUSA + ENCODING OTIMIZADO
[STATS] Dataset Original: 121811 incidentes
[STATS] Outlier Detection - Removidos: 60 (0.05%)
        Mantidos: 121751 (99.95%)
[OK] Features CAUSA validas: 14
[INFO] Encoding Ciclico para Features Temporais
[INFO] Frequency Encoding para Alta Cardinalidade
[INFO] One-Hot Encoding para Baixa Cardinalidade
[INFO] StandardScaler Normalization
[OK] Features Finais: 69 (reducao de 89.9%)


In [15]:
# ===== [4b] VISUALIZACAO: Feature Selection Impact =====
print('[INFO] VISUALIZACAO: Feature Selection Impact')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Feature Selection Impact: CAUSA + Encoding Otimizado', fontsize=14, fontweight='bold')

# [1] Features Antes vs Depois
features_comparison = ['Antes (OHE Explosion)', 'Depois (CAUSA only)']
features_count = [680, X_scaled.shape[1]]
colors_comp = ['#E53935', '#43A047']
bars = axes[0].bar(features_comparison, features_count, color=colors_comp, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Numero de Features', fontweight='bold')
axes[0].set_title('1. Reducao de Features')
axes[0].grid(axis='y', alpha=0.3)
for bar, count in zip(bars, features_count):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(count), ha='center', va='bottom', fontweight='bold', fontsize=11)

# [2] Pie Chart
removed_count = 680 - X_scaled.shape[1]
kept_count = X_scaled.shape[1]
sizes = [kept_count, removed_count]
labels_pie = ['Mantidas (%d)' % kept_count, 'Removidas (%d)' % removed_count]
colors_pie = ['#43A047', '#E53935']
axes[1].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
           startangle=90, textprops={'fontsize': 10, 'weight': 'bold'})
axes[1].set_title('2. Features Removidas vs Mantidas')

# [3] Correlacao
if X_scaled.shape[1] > 1:
    X_scaled_df = pd.DataFrame(X_scaled)
    corr_post = X_scaled_df.corr()
    corr_sample = corr_post.iloc[:min(15, len(corr_post)), :min(15, len(corr_post))]
    sns.heatmap(corr_sample, annot=False, cmap='coolwarm', center=0,
               ax=axes[2], cbar_kws={'label': 'Correlacao'}, vmin=-1, vmax=1)
    axes[2].set_title('3. Correlacao Pos-Selecao')

plt.tight_layout()
feat_selection_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'feature_selection_impact.png')
fig.savefig(feat_selection_path, dpi=100, bbox_inches='tight')
plt.close()

print('[OK] Feature Selection Impact salva')


[INFO] VISUALIZACAO: Feature Selection Impact
[OK] Feature Selection Impact salva


In [16]:
# ===== [4] FEATURE ENGINEERING: SELEÇÃO CAUSA + ENCODING OTIMIZADO =====
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA

print('\n🔧 FEATURE ENGINEERING: CAUSA + ENCODING OTIMIZADO')

# ===== [4.1] OUTLIER REMOVAL (OR LÓGICO) =====
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['incident_id', 'cluster', 'data_abertura']]

print(f'\n📊 Dataset Original: {len(df):,} incidentes')
print(f'   Colunas numéricas para outlier detection: {len(numeric_cols)}')

# Calcular quantas colunas são outlier por linha (estratégia OR)
outlier_counts = pd.DataFrame(index=df.index)
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_counts[col] = ~((df[col] >= lower) & (df[col] <= upper))

outlier_total_per_row = outlier_counts.sum(axis=1)
max_outliers_allowed = len(numeric_cols) * 0.5
outlier_mask = outlier_total_per_row <= max_outliers_allowed

df_clean = df[outlier_mask].copy()
n_removed = len(df) - len(df_clean)

print(f'\n🔍 Outlier Detection (IQR — OR lógico):' + 
      f'\n   Removidos: {n_removed:,} incidentes ({n_removed/len(df)*100:.2f}%)' +
      f'\n   Mantidos: {len(df_clean):,} incidentes ({len(df_clean)/len(df)*100:.2f}%)')

# ===== [4.2] SELECIONAR APENAS FEATURES CAUSA =====
cause_cols = [
    'prioridade_num', 'grupo_designado', 'categoria', 'subcategoria', 'produto',
    'hora_abertura', 'turno_abertura', 'dia_semana_num', 'fora_horario_comercial',
    'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'triagem_incompleta'
]

# Verificar quais existem no dataframe
cause_cols_valid = [c for c in cause_cols if c in df_clean.columns]
print(f'\n✅ Features CAUSA válidas: {len(cause_cols_valid)} (original: {len(cause_cols)})')
print(f'   {cause_cols_valid}')

df_features = df_clean[cause_cols_valid].copy()

# ===== [4.3] ENCODING CÍCLICO PARA TEMPORAIS =====
print(f'\n🔄 Encoding Cíclico para Features Temporais...')
temporal_cyclic = {
    'hora_abertura': 24,    # Hora 23 ≈ Hora 0
    'dia_semana_num': 7,    # Domingo ≈ Sábado
    'mes_abertura': 12      # Dezembro ≈ Janeiro
}

for col, period in temporal_cyclic.items():
    if col in df_features.columns:
        df_features[f'{col}_sin'] = np.sin(2 * np.pi * df_features[col] / period)
        df_features[f'{col}_cos'] = np.cos(2 * np.pi * df_features[col] / period)
        df_features = df_features.drop(columns=[col])
        print(f'   ✅ {col} → {col}_sin, {col}_cos')

# ===== [4.4] FREQUENCY ENCODING PARA ALTA CARDINALIDADE =====
print(f'\n🏷️  Frequency Encoding para Alta Cardinalidade...')
high_cardinality_cols = ['grupo_designado', 'subcategoria', 'produto']

for col in high_cardinality_cols:
    if col in df_features.columns:
        freq_map = df_features[col].value_counts(normalize=True).to_dict()
        df_features[col] = df_features[col].map(freq_map).fillna(0)
        print(f'   ✅ {col} → Frequency-encoded ({len(freq_map)} unique values → 1 feature)')

# ===== [4.5] ONE-HOT ENCODING PARA BAIXA CARDINALIDADE =====
print(f'\n📦 One-Hot Encoding para Baixa Cardinalidade...')
low_cardinality_cols = ['categoria', 'turno_abertura']
low_cardinality_cols = [c for c in low_cardinality_cols if c in df_features.columns]

if low_cardinality_cols:
    # Preencher NaN com 'unknown'
    for col in low_cardinality_cols:
        df_features[col] = df_features[col].fillna('unknown').astype(str)
    
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', max_categories=50)
    X_ohe = encoder.fit_transform(df_features[low_cardinality_cols])
    feature_names_ohe = encoder.get_feature_names_out(low_cardinality_cols)
    X_ohe_df = pd.DataFrame(X_ohe, columns=feature_names_ohe, index=df_features.index)
    
    # Remover colunas originais e adicionar OHE
    df_features = df_features.drop(columns=low_cardinality_cols)
    df_features = pd.concat([df_features, X_ohe_df], axis=1)
    
    print(f'   ✅ {low_cardinality_cols} → {X_ohe.shape[1]} features OHE')

# ===== [4.6] STANDARDSCALER =====
print(f'\n⚖️  StandardScaler Normalization...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features.fillna(0))

print(f'\n✅ Features Finais:')
print(f'   Antes (OHE explosion): 680 features')
print(f'   Depois (CAUSA only + encoding otimizado): {X_scaled.shape[1]} features')
print(f'   Redução: {(1 - X_scaled.shape[1]/680)*100:.1f}%')
print(f'   Shape: {X_scaled.shape}')
print(f'\n✅ Preparação completa!')



🔧 FEATURE ENGINEERING: CAUSA + ENCODING OTIMIZADO

📊 Dataset Original: 121,811 incidentes
   Colunas numéricas para outlier detection: 17

🔍 Outlier Detection (IQR — OR lógico):
   Removidos: 60 incidentes (0.05%)
   Mantidos: 121,751 incidentes (99.95%)

✅ Features CAUSA válidas: 14 (original: 14)
   ['prioridade_num', 'grupo_designado', 'categoria', 'subcategoria', 'produto', 'hora_abertura', 'turno_abertura', 'dia_semana_num', 'fora_horario_comercial', 'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'triagem_incompleta']

🔄 Encoding Cíclico para Features Temporais...
   ✅ hora_abertura → hora_abertura_sin, hora_abertura_cos
   ✅ dia_semana_num → dia_semana_num_sin, dia_semana_num_cos
   ✅ mes_abertura → mes_abertura_sin, mes_abertura_cos

🏷️  Frequency Encoding para Alta Cardinalidade...
   ✅ grupo_designado → Frequency-encoded (17 unique values → 1 feature)
   ✅ subcategoria → Frequency-encoded (446 unique values → 1 feature)
   ✅ produto → Frequency-encoded (51 

In [17]:
# ===== [4b] VISUALIZAÇÃO: IMPACTO DA SELEÇÃO DE FEATURES =====
print('\n📊 VISUALIZAÇÃO: Feature Selection Impact')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Feature Selection Impact: CAUSA + Encoding Otimizado', fontsize=14, fontweight='bold')

# [1] Barplot: Features Antes vs Depois
features_comparison = ['Antes (OHE Explosion)', 'Depois (CAUSA only)']
features_count = [680, X_scaled.shape[1]]
colors_comp = ['#E53935', '#43A047']
bars = axes[0].bar(features_comparison, features_count, color=colors_comp, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Número de Features', fontweight='bold')
axes[0].set_title('1. Redução de Features')
axes[0].grid(axis='y', alpha=0.3)
for bar, count in zip(bars, features_count):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{count}\nfeatures', ha='center', va='bottom', fontweight='bold', fontsize=11)

# [2] Pie Chart: Qual % foi removido e por quê
removed_count = 680 - X_scaled.shape[1]
kept_count = X_scaled.shape[1]
sizes = [kept_count, removed_count]
labels_pie = [f'Mantidas\n({kept_count})\nCAUSA only', f'Removidas\n({removed_count})\nEFEITO, OHE\nexpl., NULL']
colors_pie = ['#43A047', '#E53935']
explode = (0.05, 0.1)
axes[1].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
           explode=explode, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('2. Features Removidas vs Mantidas')

# [3] Heatmap de Correlação Pós-Seleção (sample)
# Reconstruir dataframe com feature nomes para análise
if X_scaled.shape[1] <= 50:  # Se temos poucos features, mostrar todos
    X_scaled_df = pd.DataFrame(X_scaled)
    corr_post = X_scaled_df.corr()
    # Mostrar apenas algumas features para legibilidade
    sns.heatmap(corr_post.iloc[:15, :15], annot=False, cmap='coolwarm', center=0,
               ax=axes[2], cbar_kws={'label': 'Correlação'}, vmin=-1, vmax=1)
    axes[2].set_title('3. Correlação Pós-Seleção (primeiras 15 features)')
else:
    X_scaled_df = pd.DataFrame(X_scaled)
    corr_post = X_scaled_df.corr()
    sns.heatmap(corr_post.iloc[:20, :20], annot=False, cmap='coolwarm', center=0,
               ax=axes[2], cbar_kws={'label': 'Correlação'}, vmin=-1, vmax=1)
    axes[2].set_title('3. Correlação Pós-Seleção (primeiras 20 features)')

plt.tight_layout()
feat_selection_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'feature_selection_impact.png')
fig.savefig(feat_selection_path, dpi=100, bbox_inches='tight')
plt.close()

print(f'\n✅ Feature Selection Impact salva em: {feat_selection_path}')



📊 VISUALIZAÇÃO: Feature Selection Impact

✅ Feature Selection Impact salva em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans\feature_selection_impact.png


In [18]:
# ===== [4c] PCA: REDUÇÃO DIMENSIONAL (95% variância) =====
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
n_components = pca.n_components_
explained_var = sum(pca.explained_variance_ratio_)

print(f'\n📉 PCA: REDUÇÃO DIMENSIONAL')
print(f'   Features antes: {X_scaled.shape[1]}')
print(f'   Componentes após: {n_components}')
print(f'   Variância explicada: {explained_var*100:.2f}%')
print(f'   Shape pós-PCA: {X_pca.shape}')


📉 PCA: REDUÇÃO DIMENSIONAL
   Features antes: 69
   Componentes após: 55
   Variância explicada: 95.11%
   Shape pós-PCA: (121751, 55)


In [19]:
# ===== [5] K-MEANS TRAINING (no espaço PCA) =====
k = 4

# Verificar se PCA foi executado (célula 4c)
if 'X_pca' not in locals():
    print('⚠️ Aviso: X_pca não definido. Usando X_scaled diretamente.')
    X_train = X_scaled
    n_components = X_scaled.shape[1]
else:
    X_train = X_pca
    if 'n_components' not in locals():
        n_components = X_pca.shape[1]

model = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
print(f'\n🤖 Treinando K-Means com k={k} (espaço com {n_components} dimensões)...')

try:
    labels = model.fit_predict(X_train)
    
    sil_score = silhouette_score(X_train, labels)
    db_score = davies_bouldin_score(X_train, labels)
    
    print(f'\n📊 Métricas de Clustering:')
    print(f'   Silhouette Score: {sil_score:.4f}')
    print(f'   Davies-Bouldin Index: {db_score:.4f}')
    print(f'   Dataset: {len(labels):,} amostras')
    print(f'   ✅ Modelo KMeans treinado com sucesso')
except Exception as e:
    print(f'❌ Erro ao treinar KMeans: {str(e)}')
    print(f'   Verifique se as células anteriores foram executadas corretamente')
    raise


🤖 Treinando K-Means com k=4 (espaço com 55 dimensões)...

📊 Métricas de Clustering:
   Silhouette Score: 0.0720
   Davies-Bouldin Index: 2.5110
   Dataset: 121,751 amostras
   ✅ Modelo KMeans treinado com sucesso


In [ ]:
# ===== [6a] ATRIBUIÇÃO DE CLUSTERS (limpos + outliers) =====
# Adicionar clusters ao dataset limpo
df_clean['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
df_clean['cluster_label'] = df_clean['cluster_pred'].map(cluster_names)

print(f'\n✅ Clusters atribuídos ao dataset limpo ({len(df_clean):,} registros)')

# ===== PREDIZER CLUSTERS PARA OS OUTLIERS REMOVIDOS =====
df_outliers = df[~outlier_mask].copy() if 'outlier_mask' in locals() else pd.DataFrame()

if len(df_outliers) > 0:
    print(f'\n🔄 Atribuindo clusters aos {len(df_outliers):,} incidentes removidos como outliers...')
    
    try:
        # Define as colunas categoricas que foram usadas para OneHotEncoding
        low_cardinality_cols = ['categoria', 'turno_abertura']
        high_cardinality_cols = ['grupo_designado', 'subcategoria', 'produto']
        temporal_cyclic_cols = ['hora_abertura', 'dia_semana_num', 'mes_abertura']
        
        # Preparar features dos outliers (mesmo processo que df_clean)
        X_outliers = df_outliers[cause_cols_valid].copy() if 'cause_cols_valid' in locals() else df_outliers.copy()
        
        # 1. Cyclical encoding para features temporais
        for col, period in zip(temporal_cyclic_cols, [24, 7, 12]):
            if col in X_outliers.columns:
                X_outliers[f'{col}_sin'] = np.sin(2 * np.pi * X_outliers[col] / period)
                X_outliers[f'{col}_cos'] = np.cos(2 * np.pi * X_outliers[col] / period)
        
        # 2. Frequency encoding para alta cardinalidade (calculado a partir de df_clean)
        for col in high_cardinality_cols:
            if col in X_outliers.columns:
                # Usar frequências do dataset limpo
                freq_map = df_clean[col].value_counts(normalize=True).to_dict()
                X_outliers[col] = X_outliers[col].map(freq_map).fillna(0)
        
        # 3. One-Hot Encoding com mesmo encoder
        if 'encoder' in locals() and low_cardinality_cols:
            for col in low_cardinality_cols:
                if col in X_outliers.columns:
                    X_outliers[col] = X_outliers[col].fillna('unknown').astype(str)
            
            X_outliers_cat = encoder.transform(X_outliers[low_cardinality_cols])
            X_outliers_cat_df = pd.DataFrame(X_outliers_cat, columns=encoder.get_feature_names_out(low_cardinality_cols))
            
            # Selecionar apenas as colunas numericas que foram usadas
            numeric_cols_use = [c for c in X_outliers.columns if c not in low_cardinality_cols and 
                               X_outliers[c].dtype in ['int64', 'float64']]
            X_outliers_num = X_outliers[numeric_cols_use].reset_index(drop=True)
            X_outliers_proc = pd.concat([X_outliers_num, X_outliers_cat_df.reset_index(drop=True)], axis=1)
        else:
            X_outliers_proc = X_outliers
        
        # 4. Escalar com mesmo scaler
        if 'scaler' in locals():
            X_outliers_scaled = scaler.transform(X_outliers_proc.fillna(0))
        else:
            X_outliers_scaled = X_outliers_proc.fillna(0)
        
        # 5. Aplicar PCA com mesmo transformer
        if 'pca' in locals() and 'X_pca' in locals():
            X_outliers_pca = pca.transform(X_outliers_scaled)
            labels_outliers = model.predict(X_outliers_pca)
        else:
            labels_outliers = model.predict(X_outliers_scaled)
        
        df_outliers['cluster_pred'] = labels_outliers
        df_outliers['cluster_label'] = df_outliers['cluster_pred'].map(cluster_names)
        
        print(f'   ✅ Clusters preditos para outliers')
    except Exception as e:
        print(f'   ⚠️ Erro ao predizer clusters para outliers: {str(e)}')
        print(f'   Continuando apenas com dataset limpo')
        df_outliers = pd.DataFrame()
else:
    print('   ℹ️ Nenhum outlier removido, pulando etapa')

# ===== COMBINAR TODOS OS INCIDENTES =====
if len(df_outliers) > 0:
    df_final = pd.concat([df_clean, df_outliers], ignore_index=False).sort_index()
else:
    df_final = df_clean.copy()

print(f'\n📋 Distribuição Final de Clusters:')
cluster_dist = df_final['cluster_label'].value_counts().sort_index()
for label in ['A', 'B', 'C', 'D']:
    count = cluster_dist.get(label, 0)
    pct = count / len(df_final) * 100 if count > 0 else 0
    print(f'   Cluster {label}: {count:>8,} incidentes ({pct:>5.2f}%)')

print(f'\n✅ Dataset Final:')
print(f'   Total: {len(df_final):,} incidentes (100.00%)')
print(f'   Todos os incidentes receberam cluster label')

In [ ]:
# ===== [6b] CLUSTER PROFILING: INTERPRETAR COM FEATURES EFEITO =====
print('[INFO] CLUSTER PROFILING: Interpretacao dos Clusters')

# Features de EFEITO para interpretar clusters
effect_cols = [
    'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado',
    'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional'
]

# Usar colunas de EFEITO que já existem em df_final (herdadas de df_clean)
effect_cols_available = [c for c in effect_cols if c in df_final.columns]
df_with_effect = df_final.copy()

# Criar visualizacao com 4 graficos
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Cluster Profiling: Interpretacao com Features EFEITO', fontsize=14, fontweight='bold')

colors_map = {'A': '#E53935', 'B': '#1E88E5', 'C': '#43A047', 'D': '#8E24AA'}

# [1] Tamanho dos clusters
cluster_sizes = df_final['cluster_label'].value_counts().sort_index()
bars = axes[0, 0].bar(cluster_sizes.index, cluster_sizes.values,
                       color=[colors_map[c] for c in cluster_sizes.index],
                       edgecolor='black', linewidth=2)
axes[0, 0].set_ylabel('Numero de Incidentes')
axes[0, 0].set_title('1. Distribuicao de Clusters')
axes[0, 0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, cluster_sizes.values):
    pct = val/len(df_final)*100
    axes[0, 0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
                   f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

# [2] Tempo medio de resolucao por cluster
if 'horas_ate_resolucao' in df_with_effect.columns:
    df_with_effect.boxplot(column='horas_ate_resolucao', by='cluster_label', ax=axes[0, 1])
    axes[0, 1].set_xlabel('Cluster')
    axes[0, 1].set_ylabel('Horas ate Resolucao')
    axes[0, 1].set_title('2. Tempo de Resolucao por Cluster')
    axes[0, 1].get_figure().suptitle('')  # Remove o titulo automatico

# [3] Perfil de metricas EFEITO por cluster (heatmap normalizado)
numeric_effect = [c for c in effect_cols_available if c in df_with_effect.columns and df_with_effect[c].dtype in ['int64', 'float64']]
if numeric_effect:
    profile = df_with_effect.groupby('cluster_label')[numeric_effect].mean()
    profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)
    sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[1, 0],
               cbar_kws={'label': 'Score Normalizado'})
    axes[1, 0].set_title('3. Perfil de Metricas EFEITO (normalizado)')
    axes[1, 0].set_ylabel('Cluster')

# [4] Taxa de resolucao por cluster
if 'foi_resolvido' in df_with_effect.columns:
    resolution_rate = df_with_effect.groupby('cluster_label')['foi_resolvido'].apply(lambda x: (x.sum()/len(x)*100))
    bars = axes[1, 1].bar(resolution_rate.index, resolution_rate.values,
                          color=[colors_map[c] for c in resolution_rate.index],
                          edgecolor='black', linewidth=2)
    axes[1, 1].set_ylabel('Taxa de Resolucao (%)')
    axes[1, 1].set_title('4. Taxa de Resolucao por Cluster')
    axes[1, 1].set_ylim([0, 105])
    axes[1, 1].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, resolution_rate.values):
        axes[1, 1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                       f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
cluster_profile_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'cluster_profile_efeito.png')
fig.savefig(cluster_profile_path, dpi=100, bbox_inches='tight')
plt.close()

print('[OK] Cluster Profiling salvo')

# Mostrar resumo interpretativo
print('[INFO] INTERPRETACAO DOS CLUSTERS:')
for label in sorted(df_final['cluster_label'].unique()):
    mask = df_final['cluster_label'] == label
    count = mask.sum()
    pct = count/len(df_final)*100
    print(f'  Cluster {label}: {count:,} incidentes ({pct:.1f}%)')
    
    if 'horas_ate_resolucao' in df_with_effect.columns:
        avg_duration = df_with_effect[mask]['horas_ate_resolucao'].mean()
        print(f'           Tempo medio: {avg_duration:.1f} horas')
    
    if 'excedeu_tempo_esperado' in df_with_effect.columns:
        sla_violation = df_with_effect[mask]['excedeu_tempo_esperado'].mean() * 100
        print(f'           Taxa SLA violado: {sla_violation:.1f}%')

In [ ]:
# ===== [7] SALVAR RESULTADOS (todos os incidentes) =====
from pathlib import Path

base_path = Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans')
base_path.mkdir(parents=True, exist_ok=True)

print(f'\n💾 Salvando resultados em: {base_path}')

# Salvar atribuição de clusters para TODOS os incidentes
if 'df_final' in locals() and len(df_final) > 0:
    cluster_results = df_final[['incident_id', 'cluster_pred', 'cluster_label']].copy()
    cluster_results.to_csv(str(base_path / 'kmeans_cluster_assignments.csv'), index=False)
    print(f'   ✅ kmeans_cluster_assignments.csv ({len(cluster_results):,} registros)')

    # Salvar estatísticas por cluster das colunas numéricas
    num_cols_stats = [c for c in num_cols if c in df_final.columns] if 'num_cols' in locals() else []
    if num_cols_stats:
        profile_stats = df_final.groupby('cluster_label')[num_cols_stats].agg(['mean', 'std', 'min', 'max'])
        profile_stats.to_csv(str(base_path / 'kmeans_cluster_profile_stats.csv'))
        print(f'   ✅ kmeans_cluster_profile_stats.csv')

    # Resumo de métricas
    summary_data = {
        'métrica': [
            'Modelo',
            'N Clusters',
            'Silhouette Score',
            'Davies-Bouldin Index',
            'Componentes PCA',
            'Registros processados',
            'Registros removidos',
            'Total final'
        ],
        'valor': [
            'K-Means',
            k,
            f'{sil_score:.4f}' if 'sil_score' in locals() else 'N/A',
            f'{db_score:.4f}' if 'db_score' in locals() else 'N/A',
            n_components if 'n_components' in locals() else 'N/A',
            len(df_clean) if 'df_clean' in locals() else 0,
            len(df_outliers) if 'df_outliers' in locals() else 0,
            len(df_final)
        ]
    }
    summary = pd.DataFrame(summary_data)
    summary.to_csv(str(base_path / 'kmeans_summary.csv'), index=False)
    print(f'   ✅ kmeans_summary.csv')

    # Distribuição dos clusters (detalhado)
    cluster_dist = df_final['cluster_label'].value_counts().sort_index()
    dist_df = pd.DataFrame({
        'cluster': cluster_dist.index,
        'count': cluster_dist.values,
        'percentage': (cluster_dist.values / len(df_final) * 100).round(2)
    })
    dist_df.to_csv(str(base_path / 'kmeans_cluster_distribution.csv'), index=False)
    print(f'   ✅ kmeans_cluster_distribution.csv')

    print(f'\n✅ Todos os resultados salvos com sucesso!')
else:
    print('   ❌ Erro: df_final não definido ou vazio')


💾 Salvando resultados em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans
   ✅ kmeans_cluster_assignments.csv (121,811 registros)
   ✅ kmeans_cluster_profile_stats.csv
   ✅ kmeans_summary.csv
   ✅ kmeans_cluster_distribution.csv

✅ Todos os resultados salvos com sucesso!


In [ ]:
# ===== [8b] RASTREAMENTO COMPLETO COM MLFLOW =====
mlflow.set_experiment('kmeans_clustering')
with mlflow.start_run(run_name="kmeans_v2_pca_k4_balanced"):
    
    # ===== PARÂMETROS DO MODELO =====
    mlflow.log_params({
        'model_type': 'KMeans',
        'n_clusters': k,
        'random_state': 42,
        'n_init': 10,
        'max_iter': 300,
        'outlier_detection_method': 'IQR (OR lógico)',
        'outlier_removal_percentage': round(n_removed / len(df) * 100, 2) if n_removed > 0 else 0,
        'outlier_threshold_columns': round(len(numeric_cols) * 0.5, 1),
        'features_before_encoding': X_scaled.shape[1] if 'X_scaled' in locals() else 'N/A',
        'features_before_pca': X_scaled.shape[1] if 'X_scaled' in locals() else 'N/A',
        'features_after_pca': n_components,
        'pca_variance_threshold': 0.95,
        'records_cleaned': len(df_clean),
        'records_outliers': len(df_outliers) if len(df_outliers) > 0 else 0,
        'records_total': len(df_final),
        'normalization': 'StandardScaler'
    })

    # ===== MÉTRICAS DE QUALIDADE =====
    mlflow.log_metrics({
        'silhouette_score': float(sil_score),
        'davies_bouldin_index': float(db_score),
        'n_samples': len(df_final),
        'n_features_final': n_components,
        'n_features_original': X_scaled.shape[1] if 'X_scaled' in locals() else 0
    })

    # Distribuição dos clusters
    for label in sorted(df_final['cluster_label'].unique()):
        count = (df_final['cluster_label'] == label).sum()
        percentage = count / len(df_final) * 100
        mlflow.log_metric(f'cluster_{label}_count', int(count))
        mlflow.log_metric(f'cluster_{label}_percentage', round(percentage, 2))

    # ===== ARTEFATOS: MODELO =====
    model_path = os.path.join(tempfile.gettempdir(), 'kmeans_model.pkl')
    joblib.dump(model, model_path)
    mlflow.log_artifact(model_path, 'model')

    # Salvar scaler
    scaler_path = os.path.join(tempfile.gettempdir(), 'standard_scaler.pkl')
    joblib.dump(scaler, scaler_path)
    mlflow.log_artifact(scaler_path, 'preprocessing')

    # Salvar PCA
    pca_model_path = os.path.join(tempfile.gettempdir(), 'pca_95_var.pkl')
    joblib.dump(pca, pca_model_path)
    mlflow.log_artifact(pca_model_path, 'preprocessing')

    # ===== ARTEFATOS: GRÁFICOS =====
    mlflow.log_artifact(elbow_path, 'evaluation')

    # PCA Visualization 2D
    pca_2d = PCA(n_components=2, random_state=42)
    X_2d = pca_2d.fit_transform(X_train)
    
    fig_pca, ax_pca = plt.subplots(figsize=(10, 8))
    colors_map = {'A': '#E53935', 'B': '#1E88E5', 'C': '#43A047', 'D': '#8E24AA'}
    
    for label in sorted(df_final['cluster_label'].unique()):
        mask = df_final['cluster_label'].values == label
        ax_pca.scatter(X_2d[mask, 0], X_2d[mask, 1], 
                      c=colors_map[label], label=f'Cluster {label}', 
                      alpha=0.6, s=30, edgecolors='k', linewidth=0.5)
    
    ax_pca.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} var)')
    ax_pca.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} var)')
    ax_pca.set_title(f'K-Means Clustering — PCA 2D (k={k}, Silhouette={sil_score:.4f})')
    ax_pca.legend()
    ax_pca.grid(True, alpha=0.3)
    
    pca_path = os.path.join(tempfile.gettempdir(), 'pca_clusters_2d.png')
    fig_pca.savefig(pca_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(pca_path, 'evaluation')
    plt.close(fig_pca)

    # Distribuição dos clusters
    fig_dist, ax_dist = plt.subplots(figsize=(10, 5))
    cluster_counts_final = df_final['cluster_label'].value_counts().sort_index()
    bars = ax_dist.bar(cluster_counts_final.index, cluster_counts_final.values, 
                       color=[colors_map[c] for c in cluster_counts_final.index])
    ax_dist.set_xlabel('Cluster Label')
    ax_dist.set_ylabel('Number of Incidents')
    ax_dist.set_title(f'Cluster Distribution (k={k}, Silhouette={sil_score:.4f}, Total={len(df_final):,})')
    ax_dist.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, cluster_counts_final.values):
        pct = val/len(df_final)*100
        ax_dist.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
                    f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')
    
    dist_path = os.path.join(tempfile.gettempdir(), 'cluster_distribution.png')
    fig_dist.savefig(dist_path, dpi=100, bbox_inches='tight')
    mlflow.log_artifact(dist_path, 'evaluation')
    plt.close(fig_dist)

    # ===== METADADOS E TAGS =====
    mlflow.set_tag('model_type', 'KMeans')
    mlflow.set_tag('task', 'Unsupervised_Clustering')
    mlflow.set_tag('data_version', f'{pd.Timestamp.now().date()}_gold_ml_dataset')
    mlflow.set_tag('notebook', '07_model_clustering_kmeans')
    mlflow.set_tag('evaluation_metrics', 'Silhouette_Score + Davies_Bouldin_Index')
    mlflow.set_tag('preprocessing', 'IQR_OR_lógico + OneHotEncoding + StandardScaler + PCA')
    mlflow.set_tag('outlier_handling', 'IQR_OR_method_with_prediction')
    mlflow.set_tag('dimensionality_reduction', 'PCA_95_variance')
    mlflow.set_tag('feature_count_original', str(X_scaled.shape[1]) if 'X_scaled' in locals() else '0')
    mlflow.set_tag('feature_count_pca', str(n_components))

    print(f'\n✅ MLflow Tracking Completo!')
    print(f'   Run name: kmeans_v2_pca_k4_balanced')
    print(f'   Métricas: Silhouette={sil_score:.4f}, Davies-Bouldin={db_score:.4f}')
    print(f'   Distribuição Final:')
    for label in sorted(df_final['cluster_label'].unique()):
        count = (df_final['cluster_label'] == label).sum()
        pct = count / len(df_final) * 100
        print(f'      Cluster {label}: {count:>8,} incidentes ({pct:>5.2f}%)')
    print(f'   Artefatos: Modelo KMeans, Scaler, PCA, Elbow Method, PCA 2D, Distribuição')
    print(f'   Tags: model_type, task, data_version, preprocessing, outlier_handling, etc')